In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [ ]:
#!/bin/bash
!kaggle datasets download shaunthesheep/microsoft-catsvsdogs-dataset

Dataset URL: https://www.kaggle.com/datasets/shaunthesheep/microsoft-catsvsdogs-dataset
License(s): other
 99% 782M/788M [00:03<00:00, 177MB/s]
100% 788M/788M [00:03<00:00, 222MB/s]


In [ ]:
import zipfile
zip_ref = zipfile.ZipFile('/content/microsoft-catsvsdogs-dataset.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()

In [18]:
import os
from PIL import Image

# Define your dataset path
dataset_dir = '/content/PetImages'
extensions = ['jpg', 'jpeg', 'png', 'bmp', 'gif']

# Walk through all folders
for root, dirs, files in os.walk(dataset_dir):
    for filename in files:
        file_path = os.path.join(root, filename)

        try:
            # 1. Check if file is empty (0 bytes)
            if os.path.getsize(file_path) == 0:
                print(f"Deleting empty file: {file_path}")
                os.remove(file_path)
                continue

            # 2. Check if file is a valid image using PIL
            # Only check files with image extensions
            if any(filename.lower().endswith(ext) for ext in extensions):
                try:
                    img = Image.open(file_path)
                    img.verify() # Verify integrity
                    img.close()
                    # Re-open to check if it can actually be loaded (verify() doesn't catch everything)
                    img = Image.open(file_path)
                    img.load()
                    img.close()
                except (IOError, SyntaxError) as e:
                    print(f"Deleting corrupt image: {file_path} - Error: {e}")
                    os.remove(file_path)

        except Exception as e:
            print(f"Error checking file {file_path}: {e}")

print("Cleanup complete! You can now run your dataset generators.")

Cleanup complete! You can now run your dataset generators.


In [19]:
import os
import tensorflow as tf

# Define your dataset path
data_dir = '/content/PetImages'

print("Starting strict cleanup... this may take a minute.")

# Walk through all folders
for root, dirs, files in os.walk(data_dir):
    for filename in files:
        file_path = os.path.join(root, filename)

        # specific check for the common corrupted file
        if filename == "666.jpg":
             print(f"Found known bad file {filename}... Removing.")
             os.remove(file_path)
             continue

        try:
            # Load the file as raw bytes
            img_bytes = tf.io.read_file(file_path)

            # Force TensorFlow to decode it (this is where it catches the error)
            # We use expand_animations=False to handle all formats safely
            img = tf.io.decode_image(img_bytes, channels=3, expand_animations=False)

        except tf.errors.InvalidArgumentError as e:
            print(f"Deleting corrupt image (TF rejected): {file_path}")
            os.remove(file_path)
        except Exception as e:
            print(f"Deleting error file: {file_path}")
            os.remove(file_path)

print("Strict cleanup complete.")

Starting strict cleanup... this may take a minute.
Strict cleanup complete.


In [20]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Flatten
from keras.applications.vgg16 import VGG16

In [21]:
conv_base = VGG16(
    weights='imagenet',
    include_top = False,
    input_shape=(150,150,3)
)

In [22]:
conv_base.summary()

Model: "vgg16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 150, 150, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 150, 150, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 75, 75, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 75, 75, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 75, 75, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 37, 37, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 37, 37, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 37, 37, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 37, 37, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 18, 18, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 18, 18, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 18, 18, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 18, 18, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 9, 9, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 4, 4, 512)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,714,688 (56.13 MB)

 Trainable params: 14,714,688 (56.13 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
model = Sequential()

model.add(conv_base)
model.add(Flatten())
model.add(Dense(256,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

In [24]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,812,353 (64.13 MB)

 Trainable params: 16,812,353 (64.13 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
conv_base.trainable = False

In [26]:
# generators
train_ds = keras.utils.image_dataset_from_directory(
    directory = '/content/PetImages',
    labels='inferred',
    label_mode = 'int',
    batch_size=32,
    image_size=(150,150),
    validation_split=0.2,
    subset='training',
    seed = 123
)

validation_ds = keras.utils.image_dataset_from_directory(
    directory = '/content/PetImages',
    labels='inferred',
    label_mode = 'int',
    batch_size=32,
    image_size=(150,150),
    validation_split=0.2,
    subset='validation',
     seed = 123
)

Found 24990 files belonging to 2 classes.
Using 19992 files for training.
Found 24990 files belonging to 2 classes.
Using 4998 files for validation.


In [27]:
# Normalize
def process(image,label):
    image = tensorflow.cast(image/255. ,tensorflow.float32)
    return image,label

train_ds = train_ds.map(process)
validation_ds = validation_ds.map(process)

In [28]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [29]:
history = model.fit(train_ds,epochs=10,validation_data=validation_ds)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 92s 144ms/step - accuracy: 0.8529 - loss: 0.3612 - val_accuracy: 0.8976 - val_loss: 0.2298
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 80s 128ms/step - accuracy: 0.9182 - loss: 0.1948 - val_accuracy: 0.9146 - val_loss: 0.2077
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 79s 127ms/step - accuracy: 0.9333 - loss: 0.1629 - val_accuracy: 0.9190 - val_loss: 0.1955
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 79s 127ms/step - accuracy: 0.9444 - loss: 0.1370 - val_accuracy: 0.8994 - val_loss: 0.2453
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 84s 134ms/step - accuracy: 0.9482 - loss: 0.1275 - val_accuracy: 0.9156 - val_loss: 0.2142
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 80s 127ms/step - accuracy: 0.9648 - loss: 0.0909 - val_accuracy: 0.9094 - val_loss: 0.2553
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 80s 128ms/step - accuracy: 0.9698 - loss: 0.0742 - val_accuracy: 0.9050 - val_loss: 0.2775
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 84s 134ms/step - accuracy: 0.9737 - loss: 0